# SomaliTTS — Somali SpeechT5 baseline (Kaggle GPU)

This notebook tests the two candidate Somali text-to-speech checkpoints **Jelle** and **Diirow** before any voice-conversion work. It does not perform voice cloning or RVC.

## Before running
- In Kaggle, select **Accelerator → GPU**. A free GPU may be queued or unavailable.
- Internet must be enabled so the notebook can download model weights (each Somali checkpoint is about 578 MB, plus the vocoder and dataset).
- This first pass uses a public CMU ARCTIC x-vector only as a temporary speaker-conditioning vector. It is not a Somali speaker and is not a substitute for the user's own consented voice sample.
- Do not upload private recordings to Kaggle until the privacy/retention implications are separately reviewed and the speaker explicitly agrees.

The notebook resolves and prints exact Hugging Face commit SHAs at runtime so the experiment can be reproduced later.

In [ ]:
%pip install -q transformers==4.52.4 datasets==3.6.0 sentencepiece soundfile huggingface_hub

In [ ]:
import os, json, torch, soundfile as sf
from datetime import datetime, timezone
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoProcessor, AutoModelForTextToSpectrogram, SpeechT5HifiGan
from IPython.display import Audio, display

assert torch.cuda.is_available(), 'GPU not detected. In Kaggle choose Settings → Accelerator → GPU, then rerun.'
device = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('Transformers: 4.52.4')
os.makedirs('/kaggle/working/somali_baseline', exist_ok=True)

TEXT = 'Maanta waxaan tijaabinaynaa codka af Soomaaliga. Waxaan rabnaa in hadalku noqdo mid cad oo dabiici ah.'
MODEL_IDS = ['Somali-tts/Jelle_tts', 'Somali-tts/diirow_tts']
VOCODER_ID = 'microsoft/speecht5_hifigan'
api = HfApi()

# Resolve exact model revisions for a reproducible run.
resolved = {model_id: api.model_info(model_id).sha for model_id in MODEL_IDS}
resolved[VOCODER_ID] = api.model_info(VOCODER_ID).sha
run_info = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'models': resolved, 'text': TEXT}
print(json.dumps(run_info, ensure_ascii=False, indent=2))

## Speaker-conditioning vector

SpeechT5 requires a 512-dimensional speaker embedding. The public benchmark vector below lets us test whether the Somali checkpoints produce audio at all. Because it comes from CMU ARCTIC (English), this is only a technical baseline, not a fair final judgment of Somali voice quality or the target speaker's voice.

In [ ]:
embeddings = load_dataset('Matthijs/cmu-arctic-xvectors', split='validation')
speaker_embedding = torch.tensor(embeddings[7306]['xvector'], dtype=torch.float32).unsqueeze(0).to(device)
assert tuple(speaker_embedding.shape) == (1, 512), speaker_embedding.shape
print('Speaker embedding shape:', tuple(speaker_embedding.shape))

In [ ]:
vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID, revision=resolved[VOCODER_ID]).to(device).eval()
results = {}

for model_id in MODEL_IDS:
    revision = resolved[model_id]
    print(f'\nLoading {model_id} @ {revision}')
    processor = AutoProcessor.from_pretrained(model_id, revision=revision)
    model = AutoModelForTextToSpectrogram.from_pretrained(
        model_id, revision=revision, use_safetensors=True
    ).to(device).eval()
    inputs = processor(text=TEXT, return_tensors='pt')
    input_ids = inputs['input_ids'].to(device)
    with torch.inference_mode():
        speech = model.generate_speech(input_ids, speaker_embedding, vocoder=vocoder)
    audio = speech.detach().float().cpu().numpy()
    filename = model_id.split('/')[-1].lower() + '.wav'
    output_path = f'/kaggle/working/somali_baseline/{filename}'
    sf.write(output_path, audio, samplerate=16000)
    duration = len(audio) / 16000
    results[model_id] = {'revision': revision, 'file': output_path, 'sample_rate':16000, 'duration_seconds':round(duration, 2)}
    print(results[model_id])
    display(Audio(audio, rate=16000))
    del model, processor, inputs, input_ids, speech, audio
    torch.cuda.empty_cache()

run_info['outputs'] = results
with open('/kaggle/working/somali_baseline/run_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
print('\nSaved WAV files and run_manifest.json under /kaggle/working/somali_baseline')

## Evaluation checklist

Listen to both WAV files and record the result outside this notebook:
1. Are the words understandable to a native Somali listener?
2. Are Somali consonants, vowel length, and sentence rhythm preserved?
3. Are any words skipped, repeated, or pronounced as English-like sounds?
4. Which model is better, and what exact words were wrong?

**Do not move to RVC unless at least one baseline is intelligible.** The next RVC experiment must use a speaker-owned or explicitly authorized clean recording and individually audited RVC artifacts. This notebook has not been executed by the repository maintainers; a successful run and native-speaker listening review are still required.